# Forecasting: Real Operations, Demand Lab & SKU Research

**Pregunta:** ¿cuándo un ensemble mejora una predicción útil para el negocio?

Se comparan tres objetivos por separado: pedidos diarios reales para capacidad operativa; demanda semanal simulada del ejercicio Nielsen; y unidades semanales reales por SKU. Las métricas no se intercambian entre objetivos. Ejecuta primero el notebook 01 o `python main.py --source real --explain`.


## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the full repository checkout.")
sys.path.insert(0, str(ROOT/"src"))
REPORTS=ROOT/"reports"
OP_REPORTS=REPORTS/"operations"
for folder in [OP_REPORTS/"figures", OP_REPORTS/"metrics"]:
    folder.mkdir(parents=True,exist_ok=True)


**Dónde:** Jupyter/Codespaces o una copia completa del repositorio. Notebook, CLI y pruebas reutilizan los mismos módulos.

## 2. Leer el objetivo operativo real

In [2]:
from market_insights.operations import run_daily_forecast
path=ROOT/"data/processed/daily_orders.csv"
if not path.exists():
    raise FileNotFoundError("Run notebook 01 or main.py --source real first.")
daily=pd.read_csv(path,parse_dates=["date"])
cutoff=daily.date.iloc[-112]
print("Target: distinct paid UK orders per recorded day")
print("Test:",cutoff.date(),"to",daily.date.max().date())
display(daily.head())


Target: distinct paid UK orders per recorded day
Test: 2011-08-15 to 2011-12-04


,date,orders
0,2009-12-07,91.0
1,2009-12-08,100.0
2,2009-12-09,84.0
3,2009-12-10,88.0
4,2009-12-11,70.0


**Stack:** pandas. Se cuentan IDs distintos por fecha, no líneas ni unidades vendidas. Se excluyen semanas parciales de los límites. Los ceros significan que no se registraron pedidos retenidos; no prueban horarios de apertura.

## 3. Validación temporal, modelos y pesos

In [3]:
metrics,predictions,fitted,selection,validation,search,validation_predictions=run_daily_forecast(daily,cutoff)
display(pd.DataFrame([selection]))
display(validation.round(4))
display(metrics.round(4))
print("Ensemble R² target met:",metrics.set_index("model").loc["ensemble","r2"]>=0.80)


Ensemble R² target met: True


,config,ensemble_weights,selection_metric,validation_start,validation_end,validation_rows,validation_ensemble
0,balanced,"[0.8, 0.1, 0.1]",Pooled pre-test MSE,2011-05-23,2011-08-14,84,"{'mae': 7.218062585402586, 'r2': 0.80703883681..."


,fold,model,validation_start,validation_end,mae,r2,wape
0,1,weekly_naive,2011-05-23,2011-06-19,18.2857,-0.0212,0.4089
1,1,random_forest,2011-05-23,2011-06-19,9.4062,0.7556,0.2104
2,1,xgboost,2011-05-23,2011-06-19,10.0947,0.6540,0.2258
3,1,ridge,2011-05-23,2011-06-19,9.9599,0.7019,0.2227
4,1,ensemble,2011-05-23,2011-06-19,9.3586,0.7523,0.2093
5,2,weekly_naive,2011-06-20,2011-07-17,9.8214,0.5814,0.2342
6,2,random_forest,2011-06-20,2011-07-17,6.0835,0.8506,0.1451
7,2,xgboost,2011-06-20,2011-07-17,6.2456,0.8413,0.1490
8,2,ridge,2011-06-20,2011-07-17,6.7617,0.8144,0.1613
9,2,ensemble,2011-06-20,2011-07-17,5.9666,0.8518,0.1423


,model,mae,r2,wape
0,weekly_naive,11.5625,0.7357,0.1891
1,random_forest,9.9337,0.8332,0.1624
2,xgboost,10.4170,0.8272,0.1703
3,ridge,9.9292,0.8448,0.1623
4,ensemble,9.6385,0.8441,0.1576


**Stack:** RF, XGBoost y Ridge. Tres folds de 28 días anteriores al test eligen la configuración y el blend por MSE agregado. El baseline usa el mismo día de la semana anterior. Lags, medias móviles y variables de calendario son anteriores o conocidas al pronóstico. R² no es porcentaje de aciertos; MAE se mide en pedidos por día.

## 4. Gráficos y explicación del ensemble operativo

In [4]:
from market_insights.visualization import plot_daily
from market_insights.explainability import explain
plot_daily(predictions,metrics,OP_REPORTS/"figures")
shap_audit=explain(fitted,"UCI Online Retail II | United Kingdom | next-day recorded orders",OP_REPORTS/"figures",OP_REPORTS/"metrics")
display(pd.DataFrame([shap_audit]))


,sample_n,reference,ensemble_weights,postprocessing,max_additivity_error
0,80,training tree-path counts for trees; training ...,"[0.8, 0.1, 0.1]",Component forecasts floored at zero; exact cor...,0.000006


**Stack:** Matplotlib y SHAP. Contribuciones de árboles y términos lineales utilizan los pesos seleccionados. La corrección exacta para evitar predicciones negativas se exporta separadamente y entra en la comprobación de reconstrucción.

![Pedidos diarios reales](../reports/operations/figures/daily_forecast.png)

![Comparación de modelos](../reports/operations/figures/model_comparison.png)

## 5. Demanda semanal del ejercicio Nielsen — simulación

In [5]:
from market_insights.data import demo_demand
from market_insights.features import build_features,temporal_split
from market_insights.forecasting import select_ensemble,fit_forecasters,series_scores
simulation=demo_demand()
simulation.week_start=pd.to_datetime(simulation.week_start)
simulation_cutoff=pd.Timestamp("2022-01-03")
simulation_selection,simulation_validation,_,_=select_ensemble(simulation,simulation_cutoff)
features,numeric=build_features(simulation,simulation_cutoff)
train,test=temporal_split(features,simulation_cutoff)
simulation_metrics,simulation_predictions,simulation_fitted=fit_forecasters(train,test,numeric,
    config=simulation_selection["config"],weights=simulation_selection["ensemble_weights"])
display(simulation_metrics.round(4))
per_series=series_scores(simulation_predictions)
print("Median ensemble R² per series:",per_series.loc[per_series.model.eq("ensemble"),"r2"].median())


Median ensemble R² per series: 0.7945255101301312


,model,mae,r2,wape
0,naive,49.4728,0.8473,0.1024
1,rolling_4,47.9708,0.8624,0.0993
2,rolling_8,57.2553,0.8171,0.1185
3,rolling_13,67.5782,0.7585,0.1399
4,rolling_26,87.1201,0.6118,0.1804
5,random_forest,39.1249,0.9040,0.0810
6,xgboost,36.3557,0.9162,0.0753
7,ridge,30.1580,0.9382,0.0624
8,ensemble,30.7477,0.9361,0.0637


**Stack:** mismo código compartido de features/modelos. El generador original conserva sus ecuaciones, datos y semilla. Promociones, festivos e índices se consideran conocidos en esta simulación. R² global alto puede ocultar series débiles; el resultado no describe ventas Nielsen reales.

![Demanda simulada: una sola serie país/categoría](../reports/demo/figures/forecast_vs_actual.png)

## 6. Investigación de unidades por SKU — datos reales

In [6]:
weekly=pd.read_csv(ROOT/"data/processed/weekly_real.csv",parse_dates=["week_start"],dtype={"segment":"string"})
weekly_cutoff=pd.Timestamp(weekly.week_start.sort_values().unique()[-16])
sku_selection,sku_validation,_,_=select_ensemble(weekly,weekly_cutoff)
features,numeric=build_features(weekly,weekly_cutoff)
train,test=temporal_split(features,weekly_cutoff)
sku_metrics,sku_predictions,sku_fitted=fit_forecasters(train,test,numeric,
    config=sku_selection["config"],weights=sku_selection["ensemble_weights"])
display(sku_metrics.round(4))
print("Validation-selected baseline:",sku_selection["best_validation_baseline"])


Validation-selected baseline: rolling_13


,model,mae,r2,wape
0,naive,392.1510,-0.0481,0.6568
1,rolling_4,341.8802,0.2810,0.5726
2,rolling_8,316.8255,0.3127,0.5307
3,rolling_13,304.6074,0.2880,0.5102
4,rolling_26,303.7592,0.2682,0.5088
5,random_forest,388.2239,0.2574,0.6503
6,xgboost,417.2668,0.1448,0.6989
7,ridge,389.0586,0.0484,0.6517
8,ensemble,390.1654,0.2521,0.6535


**Stack:** pandas, RF/XGBoost/Ridge. Los 12 SKU se fijan antes del primer origen de validación. Medias móviles de 4/8/13/26 semanas ofrecen referencias más fuertes que repetir la última semana. En esta tarea, la media móvil gana al ensemble; conservar ese resultado y no usar el R² de pedidos diarios para representar unidades por SKU.

## 7. Decisión y límites

El caso operativo real supera el objetivo global R² ≥ 0.80, con errores distintos según el día de la semana. La simulación también lo supera, con menor desempeño por serie. La tarea real por SKU sigue sin alcanzar el objetivo. Son preguntas y unidades diferentes.

Las pruebas automatizadas cambian targets futuros y verifican que no cambien ni las features de train, ni el conjunto de SKU, ni los pesos seleccionados. La evaluación es investigación histórica; un periodo independiente y actual es necesario antes de usarla como servicio operativo. Revisar `docs/model_cards.md` y `docs/critical_review.md`.
